In [1]:
from datasets import load_dataset

dataset = load_dataset("social_bias_frames")

/Users/admin/Desktop/College/NLP/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 112900/112900 [00:00<00:00, 638838.23 examples/s]


In [3]:
len(dataset["train"])

112900

In [29]:
type(dataset["train"])

datasets.arrow_dataset.Dataset

In [4]:
import json
json_file_path = "full_data_rational.json"

with open(json_file_path, 'r') as file:
    sbic_data = json.load(file)

In [10]:
len(sbic_data["test"])

4698

In [21]:
sample_hitid = list(sbic_data["train"].keys())[2]
sample_hitid

'3OKP4QVBP2RJPCJHG0AKYDO9NUNAGD'

In [28]:
dataset["train"][0]

{'whoTarget': '0.0',
 'intentYN': '0.66',
 'sexYN': '0.0',
 'sexReason': '',
 'offensiveYN': '1.0',
 'annotatorGender': 'woman',
 'annotatorMinority': '',
 'sexPhrase': '',
 'speakerMinorityYN': '',
 'WorkerId': '-8935932304856669427',
 'HITId': '363A7XIFV4G2799C5V96YERJA9AVAM',
 'annotatorPolitics': 'liberal',
 'annotatorRace': 'white',
 'annotatorAge': '45.0',
 'post': 'RT @_LexC__: I\'m convinced that some of y\'all bitches get pregnant purposely because "birth control &amp; plan b pills" are effective &#128533;&#128056;&#9749;&#65039;',
 'targetMinority': '',
 'targetCategory': '',
 'targetStereotype': '',
 'dataSource': 't/davidson'}

In [42]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
val_df = dataset["validation"].to_pandas()

In [44]:
def get_rationaletrain(hit_id):
    if hit_id in sbic_data["train"]:
        return sbic_data["train"][hit_id]['rationale']
    else:
        return None
    
def get_rationaletest(hit_id):
    if hit_id in sbic_data["test"]:
        return sbic_data["test"][hit_id]['rationale']
    else:
        return None
    
def get_rationaleval(hit_id):
    if hit_id in sbic_data["val"]:
        return sbic_data["val"][hit_id]['rationale']
    else:
        return None

In [45]:
train_df['rationale'] = train_df['HITId'].apply(get_rationaletrain)
test_df['rationale'] = test_df['HITId'].apply(get_rationaletest)
val_df['rationale'] = val_df['HITId'].apply(get_rationaleval)

# Cut down on nulls and incorrects

In [49]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(112900, 20)
(17501, 20)
(16738, 20)


In [50]:
train_df.dropna(subset=['rationale'], inplace=True)
test_df.dropna(subset=['rationale'], inplace=True)
val_df.dropna(subset=['rationale'], inplace=True)

In [51]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(112703, 20)
(17484, 20)
(16717, 20)


In [55]:
train_df.dropna(subset=['offensiveYN'], inplace=True)
test_df.dropna(subset=['offensiveYN'], inplace=True)
val_df.dropna(subset=['offensiveYN'], inplace=True)

In [56]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(112703, 20)
(17484, 20)
(16717, 20)


In [57]:
train_df = train_df[train_df['offensiveYN'] != '']
test_df = test_df[test_df['offensiveYN'] != '']
val_df = val_df[val_df['offensiveYN'] != '']

In [58]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(110883, 20)
(17269, 20)
(16497, 20)


In [59]:
# print(train_df.dtypes)
# print()
# print(test_df.dtypes)
# print()
# print(val_df.dtypes)

train_df['offensiveYN'] = train_df['offensiveYN'].astype(float)
test_df['offensiveYN'] = test_df['offensiveYN'].astype(float)
val_df['offensiveYN'] = val_df['offensiveYN'].astype(float)

train_df['sexYN'] = train_df['sexYN'].astype(float)
test_df['sexYN'] = test_df['sexYN'].astype(float)
val_df['sexYN'] = val_df['sexYN'].astype(float)

train_df['targetStereotype'] = train_df['targetStereotype'].astype(str)
test_df['targetStereotype'] = test_df['targetStereotype'].astype(str)
val_df['targetStereotype'] = val_df['targetStereotype'].astype(str)

train_df['targetMinority'] = train_df['targetMinority'].astype(str)
test_df['targetMinority'] = test_df['targetMinority'].astype(str)
val_df['targetMinority'] = val_df['targetMinority'].astype(str)

train_df['targetCategory'] = train_df['targetCategory'].astype(str)
test_df['targetCategory'] = test_df['targetCategory'].astype(str)
val_df['targetCategory'] = val_df['targetCategory'].astype(str)

In [64]:
train_df = train_df[~((train_df['offensiveYN'] == 1.0) & ((train_df['targetStereotype'].isnull()) | (train_df['targetStereotype'] == '') | 
                                         (train_df['targetMinority'].isnull()) | (train_df['targetMinority'] == '') | 
                                         (train_df['targetCategory'].isnull()) | (train_df['targetCategory'] == '')))]
test_df = test_df[~((test_df['offensiveYN'] == 1.0) & ((test_df['targetStereotype'].isnull()) | (test_df['targetStereotype'] == '') | 
                                         (test_df['targetMinority'].isnull()) | (test_df['targetMinority'] == '') | 
                                         (test_df['targetCategory'].isnull()) | (test_df['targetCategory'] == '')))]
val_df = val_df[~((val_df['offensiveYN'] == 1.0) & ((val_df['targetStereotype'].isnull()) | (val_df['targetStereotype'] == '') | 
                                         (val_df['targetMinority'].isnull()) | (val_df['targetMinority'] == '') | 
                                         (val_df['targetCategory'].isnull()) | (val_df['targetCategory'] == '')))]

In [65]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(95819, 20)
(15367, 20)
(14651, 20)


In [66]:
train_df.head()

,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,WorkerId,HITId,annotatorPolitics,annotatorRace,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale
1,0.0,0.66,0.0,,0.5,man,,,,6347880360297734464,363A7XIFV4G2799C5V96YERJA9AVAM,mod-liberal,white,35.0,RT @_LexC__: I'm convinced that some of y'all ...,,,,t/davidson,\nThis post contains hate towards women who ma...
2,0.0,0.33,0.0,,0.5,man,,,,-7452610791699819066,363A7XIFV4G2799C5V96YERJA9AVAM,liberal,asian,23.0,RT @_LexC__: I'm convinced that some of y'all ...,,,,t/davidson,\nThis post contains hate towards women who ma...
3,1.0,1.0,0.0,,1.0,man,,,0.0,-500114755446676507,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,all stupid,t/davidson,\nThis post is hateful because it uses derogat...
4,1.0,1.0,0.0,,1.0,man,,,0.0,-500114755446676507,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,are not people but apes.,t/davidson,\nThis post is hateful because it uses derogat...
5,1.0,1.0,0.0,,1.0,woman,,,0.0,7912096326098817047,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,32.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,black people are monkeys,t/davidson,\nThis post is hateful because it uses derogat...


In [67]:
print(train_df["offensiveYN"].value_counts())

offensiveYN
0.0    46673
1.0    39154
0.5     9992
Name: count, dtype: int64


In [72]:
count_different_values = train_df.groupby('HITId')['offensiveYN'].nunique()

# Count the number of cases where offensiveYN has different values
num_cases = (count_different_values > 1).sum()

# Print the result
print("Number of cases where offensiveYN has different values:", num_cases)

count_different_values = test_df.groupby('HITId')['offensiveYN'].nunique()

# Count the number of cases where offensiveYN has different values
num_cases = (count_different_values > 1).sum()

# Print the result
print("Number of cases where offensiveYN has different values:", num_cases)

count_different_values = val_df.groupby('HITId')['offensiveYN'].nunique()

# Count the number of cases where offensiveYN has different values
num_cases = (count_different_values > 1).sum()

# Print the result
print("Number of cases where offensiveYN has different values:", num_cases)

Number of cases where offensiveYN has different values: 0
Number of cases where offensiveYN has different values: 0
Number of cases where offensiveYN has different values: 0


In [71]:
count_different_values = train_df.groupby('HITId')['offensiveYN'].nunique()
# Identify 'HITid' values where offensiveYN has different values
hitids_with_different_values = count_different_values[count_different_values > 1].index
# Drop rows with 'HITid' values in hitids_with_different_values
train_df = train_df[~train_df['HITId'].isin(hitids_with_different_values)]

count_different_values = test_df.groupby('HITId')['offensiveYN'].nunique()
# Identify 'HITid' values where offensiveYN has different values
hitids_with_different_values = count_different_values[count_different_values > 1].index
# Drop rows with 'HITid' values in hitids_with_different_values
test_df = test_df[~test_df['HITId'].isin(hitids_with_different_values)]

count_different_values = val_df.groupby('HITId')['offensiveYN'].nunique()
# Identify 'HITid' values where offensiveYN has different values
hitids_with_different_values = count_different_values[count_different_values > 1].index
# Drop rows with 'HITid' values in hitids_with_different_values
val_df = val_df[~val_df['HITId'].isin(hitids_with_different_values)]


In [73]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(80931, 20)
(13279, 20)
(12504, 20)


In [74]:
train_df.head()

,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,WorkerId,HITId,annotatorPolitics,annotatorRace,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale
1,0.0,0.66,0.0,,0.5,man,,,,6347880360297734464,363A7XIFV4G2799C5V96YERJA9AVAM,mod-liberal,white,35.0,RT @_LexC__: I'm convinced that some of y'all ...,,,,t/davidson,\nThis post contains hate towards women who ma...
2,0.0,0.33,0.0,,0.5,man,,,,-7452610791699819066,363A7XIFV4G2799C5V96YERJA9AVAM,liberal,asian,23.0,RT @_LexC__: I'm convinced that some of y'all ...,,,,t/davidson,\nThis post contains hate towards women who ma...
3,1.0,1.0,0.0,,1.0,man,,,0.0,-500114755446676507,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,all stupid,t/davidson,\nThis post is hateful because it uses derogat...
4,1.0,1.0,0.0,,1.0,man,,,0.0,-500114755446676507,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,are not people but apes.,t/davidson,\nThis post is hateful because it uses derogat...
5,1.0,1.0,0.0,,1.0,woman,,,0.0,7912096326098817047,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,32.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,black people are monkeys,t/davidson,\nThis post is hateful because it uses derogat...


In [76]:
num_unique_values = train_df['HITId'].nunique()
print("Number of unique values in 'HITId' column:", num_unique_values)
num_unique_values = test_df['HITId'].nunique()
print("Number of unique values in 'HITId' column:", num_unique_values)
num_unique_values = val_df['HITId'].nunique()
print("Number of unique values in 'HITId' column:", num_unique_values)

Number of unique values in 'HITId' column: 28863
Number of unique values in 'HITId' column: 3801
Number of unique values in 'HITId' column: 3773


In [77]:
train_df = train_df.drop_duplicates(subset=['HITId'])
test_df = test_df.drop_duplicates(subset=['HITId'])
val_df = val_df.drop_duplicates(subset=['HITId'])

In [78]:
num_unique_values = train_df['HITId'].nunique()
print("Number of unique values in 'HITId' column:", num_unique_values)
num_unique_values = test_df['HITId'].nunique()
print("Number of unique values in 'HITId' column:", num_unique_values)
num_unique_values = val_df['HITId'].nunique()
print("Number of unique values in 'HITId' column:", num_unique_values)

Number of unique values in 'HITId' column: 28863
Number of unique values in 'HITId' column: 3801
Number of unique values in 'HITId' column: 3773


In [79]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(28863, 20)
(3801, 20)
(3773, 20)


In [80]:
import pandas as pd

In [90]:
train_df.to_csv("train_dataset_final.csv")
test_df.to_csv("test_dataset_final.csv")
val_df.to_csv("val_dataset_final.csv")

In [84]:
train_df['rationale'] = train_df['rationale'].str.strip('\n')
test_df['rationale'] = test_df['rationale'].str.strip('\n')
val_df['rationale'] = val_df['rationale'].str.strip('\n')

/var/folders/1l/frvb0ph968s8wvl2gbc6mnw80000gn/T/ipykernel_31242/1247807764.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['rationale'] = train_df['rationale'].str.strip('\n')


In [87]:
train_df['rationale'] = train_df['rationale'].astype(str)
test_df['rationale'] = test_df['rationale'].astype(str)
val_df['rationale'] = val_df['rationale'].astype(str)

/var/folders/1l/frvb0ph968s8wvl2gbc6mnw80000gn/T/ipykernel_31242/1235812273.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['rationale'] = train_df['rationale'].astype(str)


In [89]:
train_df.dtypes

whoTarget             object
intentYN              object
sexYN                float64
sexReason             object
offensiveYN          float64
annotatorGender       object
annotatorMinority     object
sexPhrase             object
speakerMinorityYN     object
WorkerId              object
HITId                 object
annotatorPolitics     object
annotatorRace         object
annotatorAge          object
post                  object
targetMinority        object
targetCategory        object
targetStereotype      object
dataSource            object
rationale             object
dtype: object

In [91]:
train_df.head()

,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,WorkerId,HITId,annotatorPolitics,annotatorRace,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale
1,0.0,0.66,0.0,,0.5,man,,,,6347880360297734464,363A7XIFV4G2799C5V96YERJA9AVAM,mod-liberal,white,35.0,RT @_LexC__: I'm convinced that some of y'all ...,,,,t/davidson,This post contains hate towards women who may ...
3,1.0,1.0,0.0,,1.0,man,,,0.0,-500114755446676507,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,all stupid,t/davidson,This post is hateful because it uses derogator...
10,1.0,1.0,0.0,,1.0,man,,,1.0,2430897476320006833,3OKP4QVBP2RJPCJHG0AKYDO9NUNAGD,cons,white,32.0,I need a girl from Jamaica I can't fuck with t...,black folks,race,black women are basic,t/davidson,This post is hateful because it displays a cle...
12,1.0,1.0,0.0,,1.0,man,"white, male",,0.0,1389628790122253253,3WGCNLZJKF2KJ4WQIRGJHRS9681D1I,cons,white,31.0,Molly make bitches crazy,women,gender,women can't handle drugs,t/davidson,This post uses derogatory language and promote...
14,,0.0,0.0,,0.0,woman,,,,6927542478311092738,3HRWUH63QUWSTYOM88WYGMHKWS85NA,liberal,white,24.0,RT @rihanna: I ain't got no type...bad bitches...,,,,t/davidson,This post by Rihanna is not hateful because it...


In [92]:
train_df["offensiveYN"].value_counts()

offensiveYN
0.0    15563
1.0    10974
0.5     2326
Name: count, dtype: int64

NameError: name 'train_df' is not defined

# Adding atomic

In [2]:
import pandas as pd

In [3]:
train_df = pd.read_csv("train_dataset_final.csv")
test_df = pd.read_csv("test_dataset_final.csv")
val_df = pd.read_csv("val_dataset_final.csv")

In [4]:
print(train_df.shape)
print(test_df.shape)
print(val_df.shape)

(28863, 21)
(3801, 21)
(3773, 21)


In [9]:
test_df.head()

,Unnamed: 0,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,...,HITId,annotatorPolitics,annotatorRace,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale
0,4,0.0,0.66,0.0,NaN,0.5,woman,NaN,NaN,NaN,...,3OQQD2WO8I0X1IQRX9VDWHNJWXN3I2,mod-cons,white,36.0,Yo hoe is like a tick so she hunting for a blood,NaN,NaN,NaN,t/davidson,"This post compares a woman to a tick, a parasi..."
1,6,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,0.0,...,32PT7WK7DM9GT7A55VU25K764SF3D0,mod-liberal,white,27.0,When u hitting it from the back u gotta call h...,women,gender,sexually promiscuous women are called sluts,t/davidson,This post promotes the use of degrading langua...
2,9,NaN,0.00,0.5,refers to self as a ho,0.0,man,NaN,Studying for this bio test like a hoe,NaN,...,3ZFRE2BDQ98VC6MFH2QN3SFDZ5PXZS,cons,white,32.0,Studying for this bio test like a hoe &#128554;,NaN,NaN,NaN,t/davidson,This post is hateful because it uses derogator...
3,11,1.0,1.00,1.0,it's a euphemism for having sex with a woman,1.0,man,NaN,get some pussy,0.5,...,38F60IALAGBRT4758YCY8QRL42IT0D,mod-liberal,white,30.0,You niggas talking to these females trying to ...,women,gender,women are self-centered/egotistical,t/davidson,This post displays a clear disdain and contemp...
4,13,1.0,0.66,0.0,NaN,0.5,woman,"gender, sexuality, mental illness",NaN,0.0,...,32K26U12DNI05GCOP508OFZPVC7VDO,liberal,white,28.0,Fuck you bitches who send dudes to child suppo...,women,gender,women are petty,t/davidson,This post is promoting hate towards women who ...


In [18]:
import json
atomic_train_path = "train_with_commonsense_final.json"
atomic_test_path = "test_with_commonsense_final.json"
atomic_val_path = "val_with_commonsense_final.json"

# Assuming your JSON data is stored in a file named 'data.json'
with open(atomic_train_path, 'r') as f:
    atomic_train = json.load(f)

with open(atomic_test_path, 'r') as f:
    atomic_test = json.load(f)

with open(atomic_val_path, 'r') as f:
    atomic_val = json.load(f)

In [19]:
print(len(atomic_train.keys()))
print(len(atomic_test.keys()))
print(len(atomic_val.keys()))

28863
3801
3773


In [20]:
atomic_train = list(atomic_train.values())
atomic_test = list(atomic_test.values())
atomic_val = list(atomic_val.values())

In [21]:
print(type(atomic_train))
print(type(atomic_test))
print(type(atomic_val))

<class 'list'>
<class 'list'>
<class 'list'>


In [30]:
count = 0
for dic in atomic_train:
    if "commonsense" not in dic:
        count+=1
print(count)

648


In [14]:
train_df.head()

,Unnamed: 0,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,...,HITId,annotatorPolitics,annotatorRace,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale
0,1,0.0,0.66,0.0,NaN,0.5,man,NaN,NaN,NaN,...,363A7XIFV4G2799C5V96YERJA9AVAM,mod-liberal,white,35.0,RT @_LexC__: I'm convinced that some of y'all ...,NaN,NaN,NaN,t/davidson,This post contains hate towards women who may ...
1,3,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,0.0,...,3JTPR5MTZS6RLS3JBV4IOU0G2X35K5,liberal,white,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,all stupid,t/davidson,This post is hateful because it uses derogator...
2,10,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,1.0,...,3OKP4QVBP2RJPCJHG0AKYDO9NUNAGD,cons,white,32.0,I need a girl from Jamaica I can't fuck with t...,black folks,race,black women are basic,t/davidson,This post is hateful because it displays a cle...
3,12,1.0,1.00,0.0,NaN,1.0,man,"white, male",NaN,0.0,...,3WGCNLZJKF2KJ4WQIRGJHRS9681D1I,cons,white,31.0,Molly make bitches crazy,women,gender,women can't handle drugs,t/davidson,This post uses derogatory language and promote...
4,14,NaN,0.00,0.0,NaN,0.0,woman,NaN,NaN,NaN,...,3HRWUH63QUWSTYOM88WYGMHKWS85NA,liberal,white,24.0,RT @rihanna: I ain't got no type...bad bitches...,NaN,NaN,NaN,t/davidson,This post by Rihanna is not hateful because it...


In [34]:
import string

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

In [37]:
# oreact, xattr, xintent
oreact_train = []
xattr_train = []
xintent_train = []
i=0
for index, row in train_df.iterrows():
    i+=1
    if i%1000==0:
        print(i)
    commonsense = next((d for d in atomic_train if d.get("HITId") == row["HITId"]), None)
    if commonsense is None:
        print(row["HITId"], "not found")
        oreact_train.append("not applicable")
        xattr_train.append("not applicable")
        xintent_train.append("not applicable")
    elif "commonsense" not in commonsense:
        oreact_train.append("not applicable")
        xattr_train.append("not applicable")
        xintent_train.append("not applicable")
    else:
        oreact = commonsense["commonsense"]["oReact"]["beams"]
        xattr = commonsense["commonsense"]["xAttr"]["beams"]
        xintent = commonsense["commonsense"]["xIntent"]["beams"]
        final_oreact = [x for x in oreact if x != "none" and x != ""]
        final_xattr = [x for x in xattr if x != "none" and x != ""]
        final_xintent = [x for x in xintent if x != "none" and x != ""]
        final_oreact = [remove_punctuation(sentence) for sentence in final_oreact]
        final_xattr = [remove_punctuation(sentence) for sentence in final_xattr]
        final_xintent = [remove_punctuation(sentence) for sentence in final_xintent]
        if final_oreact!=[]:
            oreact_train.append(", ".join(final_oreact))
        else:
            oreact_train.append("not applicable")
        if final_xattr!=[]:
            xattr_train.append(", ".join(final_xattr))
        else:
            xattr_train.append("not applicable")
        if final_xintent!=[]:
            xintent_train.append(", ".join(final_xintent))
        else:
            xintent_train.append("not applicable")

train_df['oReact'] = oreact_train
train_df['xAttr'] = xattr_train
train_df['xIntent'] = xintent_train


1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000


In [38]:
train_df.head()

,Unnamed: 0,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,...,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale,oReact,xAttr,xIntent
0,1,0.0,0.66,0.0,NaN,0.5,man,NaN,NaN,NaN,...,35.0,RT @_LexC__: I'm convinced that some of y'all ...,NaN,NaN,NaN,t/davidson,This post contains hate towards women who may ...,not applicable,not applicable,not applicable
1,3,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,0.0,...,25.0,RT @iBeZo: Stupid fucking nigger LeBron. You f...,black folks,race,all stupid,t/davidson,This post is hateful because it uses derogator...,"annoyed, angry, disgusted, offended, sad, mad,...","stupid, ignorant, dumb, racist, careless, mean...","to be like them, people, get a bad name with p..."
2,10,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,1.0,...,32.0,I need a girl from Jamaica I can't fuck with t...,black folks,race,black women are basic,t/davidson,This post is hateful because it displays a cle...,"sad, happy, nervous, confused, scared, wanted,...","dumb, unskilled, stupid, careless, unlucky, lo...","to find a girlfriend , a girl, love, sex, the ..."
3,12,1.0,1.00,0.0,NaN,1.0,man,"white, male",NaN,0.0,...,31.0,Molly make bitches crazy,women,gender,women can't handle drugs,t/davidson,This post uses derogatory language and promote...,"annoyed, angry, mad, sad, irritated, upset, hu...","mean, annoying, cruel, rude, crazy, obnoxious,...","to be obnoxious, attention, them to be crazy, ..."
4,14,NaN,0.00,0.0,NaN,0.0,woman,NaN,NaN,NaN,...,24.0,RT @rihanna: I ain't got no type...bad bitches...,NaN,NaN,NaN,t/davidson,This post by Rihanna is not hateful because it...,"annoyed, offended , angry about it, sad, upset...","unlikable, dumb, mean, ignorant, bad, stupid, ...","to be mean, a person who gets his way, is none..."


In [39]:
# oreact, xattr, xintent
oreact_test = []
xattr_test = []
xintent_test = []
i=0
for index, row in test_df.iterrows():
    i+=1
    if i%1000==0:
        print(i)
    commonsense = next((d for d in atomic_test if d.get("HITId") == row["HITId"]), None)
    if commonsense is None:
        print(row["HITId"], "not found")
        oreact_test.append("not applicable")
        xattr_test.append("not applicable")
        xintent_test.append("not applicable")
    elif "commonsense" not in commonsense:
        oreact_test.append("not applicable")
        xattr_test.append("not applicable")
        xintent_test.append("not applicable")
    else:
        oreact = commonsense["commonsense"]["oReact"]["beams"]
        xattr = commonsense["commonsense"]["xAttr"]["beams"]
        xintent = commonsense["commonsense"]["xIntent"]["beams"]
        final_oreact = [x for x in oreact if x != "none" and x != ""]
        final_xattr = [x for x in xattr if x != "none" and x != ""]
        final_xintent = [x for x in xintent if x != "none" and x != ""]
        final_oreact = [remove_punctuation(sentence) for sentence in final_oreact]
        final_xattr = [remove_punctuation(sentence) for sentence in final_xattr]
        final_xintent = [remove_punctuation(sentence) for sentence in final_xintent]
        if final_oreact!=[]:
            oreact_test.append(", ".join(final_oreact))
        else:
            oreact_test.append("not applicable")
        if final_xattr!=[]:
            xattr_test.append(", ".join(final_xattr))
        else:
            xattr_test.append("not applicable")
        if final_xintent!=[]:
            xintent_test.append(", ".join(final_xintent))
        else:
            xintent_test.append("not applicable")

test_df['oReact'] = oreact_test
test_df['xAttr'] = xattr_test
test_df['xIntent'] = xintent_test


1000
2000
3000


In [40]:
test_df.head()

,Unnamed: 0,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,...,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale,oReact,xAttr,xIntent
0,4,0.0,0.66,0.0,NaN,0.5,woman,NaN,NaN,NaN,...,36.0,Yo hoe is like a tick so she hunting for a blood,NaN,NaN,NaN,t/davidson,"This post compares a woman to a tick, a parasi...","annoyed, scared, angry, disgusted, sad, worrie...","mean, aggressive, dangerous, creepy, careless,...","to kill, a dead animal, get a bite in her mout..."
1,6,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,0.0,...,27.0,When u hitting it from the back u gotta call h...,women,gender,sexually promiscuous women are called sluts,t/davidson,This post promotes the use of degrading langua...,"surprised , hurt confused, happy, upset, ann...","careless, lucky, fast, quick, aggressive, mean...","to get in a fight , she did nt want to hit her..."
2,9,NaN,0.00,0.5,refers to self as a ho,0.0,man,NaN,Studying for this bio test like a hoe,NaN,...,32.0,Studying for this bio test like a hoe &#128554;,NaN,NaN,NaN,t/davidson,This post is hateful because it uses derogator...,"annoyed, impressed, happy, disappointed, surpr...","smart, intelligent, dumb, stupid, curious, ski...","to get the test done, a good grade for school,..."
3,11,1.0,1.00,1.0,it's a euphemism for having sex with a woman,1.0,man,NaN,get some pussy,0.5,...,30.0,You niggas talking to these females trying to ...,women,gender,women are self-centered/egotistical,t/davidson,This post displays a clear disdain and contemp...,"annoyed, confused, flattered, happy, frustrate...","assertive, aggressive, mean, rude, pushy, soci...","to make a mistake, a new friend, sex with wome..."
4,13,1.0,0.66,0.0,NaN,0.5,woman,"gender, sexuality, mental illness",NaN,0.0,...,28.0,Fuck you bitches who send dudes to child suppo...,women,gender,women are petty,t/davidson,This post is promoting hate towards women who ...,"sad, angry, annoyed, bad, scared, upset, hurt,...","mean, cruel, careless, rude, selfish, aggressi...","to be a jerk, the other person to be safe, the..."


In [41]:
# oreact, xattr, xintent
oreact_val = []
xattr_val = []
xintent_val = []
i=0
for index, row in val_df.iterrows():
    i+=1
    if i%1000==0:
        print(i)
    commonsense = next((d for d in atomic_val if d.get("HITId") == row["HITId"]), None)
    if commonsense is None:
        print(row["HITId"], "not found")
        oreact_val.append("not applicable")
        xattr_val.append("not applicable")
        xintent_val.append("not applicable")
    elif "commonsense" not in commonsense:
        oreact_val.append("not applicable")
        xattr_val.append("not applicable")
        xintent_val.append("not applicable")
    else:
        oreact = commonsense["commonsense"]["oReact"]["beams"]
        xattr = commonsense["commonsense"]["xAttr"]["beams"]
        xintent = commonsense["commonsense"]["xIntent"]["beams"]
        final_oreact = [x for x in oreact if x != "none" and x != ""]
        final_xattr = [x for x in xattr if x != "none" and x != ""]
        final_xintent = [x for x in xintent if x != "none" and x != ""]
        final_oreact = [remove_punctuation(sentence) for sentence in final_oreact]
        final_xattr = [remove_punctuation(sentence) for sentence in final_xattr]
        final_xintent = [remove_punctuation(sentence) for sentence in final_xintent]
        if final_oreact!=[]:
            oreact_val.append(", ".join(final_oreact))
        else:
            oreact_val.append("not applicable")
        if final_xattr!=[]:
            xattr_val.append(", ".join(final_xattr))
        else:
            xattr_val.append("not applicable")
        if final_xintent!=[]:
            xintent_val.append(", ".join(final_xintent))
        else:
            xintent_val.append("not applicable")

val_df['oReact'] = oreact_val
val_df['xAttr'] = xattr_val
val_df['xIntent'] = xintent_val


1000
2000
3000


In [42]:
val_df.head()

,Unnamed: 0,whoTarget,intentYN,sexYN,sexReason,offensiveYN,annotatorGender,annotatorMinority,sexPhrase,speakerMinorityYN,...,annotatorAge,post,targetMinority,targetCategory,targetStereotype,dataSource,rationale,oReact,xAttr,xIntent
0,0,1.0,1.00,0.0,NaN,1.0,woman,NaN,NaN,0.0,...,29.0,Every half-breed cop in America is trying to r...,police officers,social,rape women,t/davidson,This post is hateful because it promotes harmf...,"scared, disgusted, angry, annoyed, ashamed, vi...","immoral, criminal, racist, desperate, dangerou...","to be a criminal, sex, a thrill, money, be wit..."
1,8,NaN,0.33,0.0,NaN,0.0,woman,veteran,NaN,NaN,...,31.0,You ratchet ass bitches gonna have nobody to b...,NaN,NaN,NaN,t/davidson,This post does not contain any language or ton...,"angry, annoyed, mad, sad angry, bad, hurt, up...","careless, irresponsible, mean, unlucky, unapol...","to have an effect on people, be left alone, pe..."
2,20,1.0,1.00,0.0,NaN,1.0,woman,NaN,NaN,0.5,...,31.0,RT @nostalgic_mike: Aw y'all some hoes hating ...,women,gender,"women are ""hoes""",t/davidson,This post is hateful because it is using derog...,"annoyed, sad, angry, happy, upset, bad, unhapp...","nostalgic, emotional, sad, mean, thoughtful, s...","to have some peace, sad, happy, angry, unhappy..."
3,25,1.0,1.00,0.0,NaN,1.0,man,NaN,NaN,0.0,...,23.0,i wonder why girls can call eachotha bitches t...,women,gender,are hypocritical,t/davidson,This post is hateful because it perpetuates ha...,"annoyed, angry, offended, hurt, sad, jealous, ...","mean, dumb, stupid, arrogant, jealous, curious...","to be friends with them, because they do nt li..."
4,27,0.0,0.33,0.0,NaN,0.5,woman,sexuality,NaN,NaN,...,35.0,TWITTER IS JUST LIKE REAL LIFE...filled with p...,NaN,NaN,NaN,t/davidson,This post is considered hateful because it use...,"happy, impressed, interested, surprised, enter...","realistic, normal, interested, happy, imaginat...","to be like people, be like person , nothing, a..."
